<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/main/RA3_LAB4/EXPERIENCIA_2%20/RA3_Lab_N%C2%B04_EXP2_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/Santibareiro27/Inteligencia-Computacional/blob/alex/RA3_LAB4/EXPERIENCIA_2%20/RA3_Lab_N%C2%B04_EXP2_G8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiencia 2 - El Problema de las N Reinas con Algoritmos Geneticos (DEAP)

**IC415 - Inteligencia Computacional - RA3 / Laboratorio N 4**

Este notebook resuelve el problema de las N Reinas para $N \in \{8, 20, 50\}$ mediante un
algoritmo genetico implementado sobre DEAP, con representacion por permutacion, cruce de
orden (OX) y mutacion por intercambio (swap). Incorpora un sistema de checkpoints atomicos
con restauracion del estado del generador aleatorio, corridas multi-semilla y el analisis
comparativo frente a la busqueda exhaustiva.

El contenido del notebook es deliberadamente conciso: el codigo y las figuras son la
evidencia tecnica reproducible; la interpretacion extensa corresponde al informe.

## 0. Configuracion del entorno

Instalacion de dependencias, importaciones y fijado global de semillas. El parametro
`SEMILLA_BASE` define el punto de partida; las corridas multi-semilla derivan de el de forma
determinista.

In [ ]:
# Instalacion silenciosa de DEAP (idempotente: si ya esta, no reinstala)
try:
    import deap
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "deap"], check=False)
    import deap

import os, time, pickle, random, zipfile, math
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from deap import base, creator, tools

print("DEAP:", deap.__version__)
print("NumPy:", np.__version__, "| pandas:", pd.__version__)

# Reproducibilidad global
SEMILLA_BASE = 42
random.seed(SEMILLA_BASE)
np.random.seed(SEMILLA_BASE)

# Directorio de figuras para el informe
FIGS_DIR = "figuras_informe"
os.makedirs(FIGS_DIR, exist_ok=True)
print("Figuras se guardaran en:", os.path.abspath(FIGS_DIR))

DEAP: 1.4
NumPy: 2.4.6 | pandas: 3.0.3
Figuras se guardaran en: /tmp/figuras_informe


## 1. Exploracion del problema

El problema no aporta un dataset, pero si una estructura combinatoria que conviene
caracterizar antes de optimizar. Se cuantifica el tamano del espacio de busqueda bajo dos
codificaciones y se documentan las metricas que guiaran todo el analisis.

**Espacio de busqueda.** Una asignacion ingenua (cada reina en cualquiera de las $N$ filas
de su columna) genera $N^{N}$ disposiciones. Si en cambio se representa la solucion como una
**permutacion** de $\{0,\dots,N-1\}$, el espacio se reduce a $N!$. La reduccion es la primera
y mas importante palanca de optimizacion, anterior a cualquier hiperparametro.

**Optimo conocido.** El minimo global de la funcion de fitness es $0$ (cero ataques
diagonales). Este valor se conoce de antemano y habilita un criterio de parada exacto:
detener apenas se alcanza, sin necesidad de agotar el presupuesto generacional.

In [ ]:
Ns = [8, 20, 50]

filas = []
for N in Ns:
    ingenuo = N ** N
    permut = math.factorial(N)
    filas.append({
        "N": N,
        "Espacio ingenuo (N^N)": f"{ingenuo:.3e}",
        "Espacio permutacion (N!)": f"{permut:.3e}",
        "Reduccion (veces)": f"{ingenuo / permut:.3e}",
    })

tabla_espacio = pd.DataFrame(filas)
print(tabla_espacio.to_string(index=False))
print()
print("N=8 en detalle:  N^N =", 8**8, " ->  N! =", math.factorial(8))

 N Espacio ingenuo (N^N) Espacio permutacion (N!) Reduccion (veces)
 8             1.678e+07                4.032e+04         4.161e+02
20             1.049e+26                2.433e+18         4.310e+07
50             8.882e+84                3.041e+64         2.920e+20

N=8 en detalle:  N^N = 16777216  ->  N! = 40320


**Metricas que se miden en cada corrida.**

- *Conflictos diagonales* (fitness, a minimizar; optimo $= 0$).
- *Generaciones hasta convergencia* (o tope `max_gen` si no converge).
- *Tiempo de pared* (wall-clock) por corrida.
- *Tasa de exito* sobre multiples semillas independientes.

## 2. Representacion y funcion de fitness

**Codificacion.** Un individuo es una lista `perm` de longitud $N$ que es una permutacion de
$\{0,\dots,N-1\}$. El indice $i$ es la columna y `perm[i]` es la fila de la reina en esa
columna.

Esta representacion **elimina por construccion dos de las tres restricciones del problema**:

- *Una reina por columna*: garantizada porque cada indice $i$ aparece exactamente una vez.
- *Una reina por fila*: garantizada porque, al ser permutacion, cada valor de fila aparece
  exactamente una vez.

La unica restriccion que queda viva es la de las **diagonales**. En consecuencia, **toda la
informacion del problema queda codificada en el conteo de conflictos diagonales**, y eso es
lo unico que el fitness debe medir.

**Costo del fitness.** Dos reinas en columnas $i, j$ se atacan en diagonal si
$\text{perm}[i] - i = \text{perm}[j] - j$ (diagonal principal) o
$\text{perm}[i] + i = \text{perm}[j] + j$ (diagonal secundaria). En lugar de comparar todos
los pares ($O(N^2)$), se agrupan las reinas por identificador de diagonal y se cuentan los
choques con $\binom{k}{2}$ por grupo de $k \ge 2$ reinas. El conteo es $O(N)$, lo que importa
porque el fitness se evalua poblacion $\times$ generaciones veces.

In [ ]:
def conflictos_diagonales(perm):
    '''Conteo O(N) de pares de reinas que se atacan en diagonal.
    perm[i] = fila de la reina en la columna i (perm es permutacion de 0..N-1).'''
    diag_principal = Counter()   # d1 = fila - columna
    diag_secundaria = Counter()  # d2 = fila + columna
    for col, fila in enumerate(perm):
        diag_principal[fila - col] += 1
        diag_secundaria[fila + col] += 1
    conflictos = 0
    for k in diag_principal.values():
        if k > 1:
            conflictos += k * (k - 1) // 2
    for k in diag_secundaria.values():
        if k > 1:
            conflictos += k * (k - 1) // 2
    return conflictos

**Validacion de la funcion de fitness.** Se contrasta contra casos verificables a mano: una
solucion valida conocida de $N=8$ debe dar $0$; la permutacion identidad ubica todas las
reinas sobre la misma diagonal principal y debe dar $\binom{N}{2}$.

In [ ]:
# Caso 1: solucion valida conocida de N=8 -> 0 conflictos
sol_valida_8 = [0, 4, 7, 5, 2, 6, 1, 3]
c1 = conflictos_diagonales(sol_valida_8)

# Caso 2: identidad -> todas en la diagonal principal -> C(N,2) conflictos
N = 8
identidad = list(range(N))
c2 = conflictos_diagonales(identidad)
esperado = N * (N - 1) // 2

print("Solucion valida N=8 :", c1, "(esperado 0)")
print("Identidad N=8       :", c2, "(esperado", esperado, ")")
assert c1 == 0 and c2 == esperado, "La funcion de fitness no supera la validacion"
print("\nValidacion superada.")

Solucion valida N=8 : 0 (esperado 0)
Identidad N=8       : 28 (esperado 28 )

Validacion superada.


## 3. Configuracion del toolbox de DEAP

Se define la infraestructura evolutiva: minimizacion (peso $-1$), individuo como permutacion,
cruce de orden (`cxOrdered`, compatible con permutaciones), mutacion por intercambio (swap
propio que garantiza seguir siendo permutacion) y seleccion por torneo de tamano 3.

Tanto OX como el swap **preservan la validez de la permutacion**: nunca reintroducen
conflictos de fila o columna. Por eso no se usan operadores de representacion binaria, que
romperian la codificacion.

In [ ]:
# El creator se define una unica vez (evita error si la celda se reejecuta)
if not hasattr(creator, "FitnessMin"):
    creator.create("FitnessMin", base.Fitness, weights=(-1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMin)


def mutacion_swap(individuo):
    '''Mutacion por intercambio: elige dos posiciones al azar e intercambia sus valores.
    Garantiza que el resultado sigue siendo una permutacion valida.'''
    i, j = random.sample(range(len(individuo)), 2)
    individuo[i], individuo[j] = individuo[j], individuo[i]
    return (individuo,)


def construir_toolbox(N):
    tb = base.Toolbox()
    tb.register("indices", random.sample, range(N), N)
    tb.register("individual", tools.initIterate, creator.Individual, tb.indices)
    tb.register("population", tools.initRepeat, list, tb.individual)
    tb.register("evaluate", lambda ind: (conflictos_diagonales(ind),))
    tb.register("mate", tools.cxOrdered)          # OX
    tb.register("mutate", mutacion_swap)          # swap
    tb.register("select", tools.selTournament, tournsize=3)
    return tb

# Prueba rapida del toolbox
_tb = construir_toolbox(8)
_ind = _tb.individual()
print("Individuo de ejemplo (N=8):", _ind, "| es permutacion:", sorted(_ind) == list(range(8)))
print("Fitness:", _tb.evaluate(_ind))

Individuo de ejemplo (N=8): [1, 0, 5, 2, 7, 6, 4, 3] | es permutacion: True
Fitness: (6,)


## 4. Sistema de checkpoints y motor evolutivo

El entorno de ejecucion (Colab) es efimero y puede desconectarse a mitad de una corrida
larga. Para que el proceso sea **reanudable y reproducible** se implementa un motor
generacional manual (no `eaSimple`) con checkpoints atomicos.

**Claves de reproducibilidad.** El checkpoint guarda el *estado* del generador aleatorio
(`random.getstate()` y `np.random.get_state()`), no la semilla. Al reanudar se **restaura**
ese estado y **nunca** se vuelve a sembrar: re-sembrar romperia la continuidad de la
secuencia aleatoria y, con ella, la reproducibilidad exacta.

In [ ]:
# --- Preparacion de la persistencia (Drive en Colab, carpeta local fuera de Colab) ---
def preparar_directorio_checkpoints():
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive", force_remount=False)
        ruta = "/content/drive/MyDrive/IC415_Exp2/checkpoints"
        entorno = "Colab + Google Drive"
    except Exception:
        ruta = "checkpoints"
        entorno = "Local (fuera de Colab)"
    os.makedirs(ruta, exist_ok=True)
    return ruta, entorno

CKPT_DIR, ENTORNO = preparar_directorio_checkpoints()
print("Entorno:", ENTORNO)
print("Checkpoints en:", os.path.abspath(CKPT_DIR))

Entorno: Local (fuera de Colab)
Checkpoints en: /tmp/checkpoints


In [ ]:
def ruta_checkpoint(ckpt_dir, N, seed, tag=""):
    '''El nombre incluye un `tag` de configuracion para que dos corridas con el mismo
    (N, seed) pero distinta configuracion (p. ej. AG puro vs. memetico) NO compartan
    archivo de checkpoint. Sin esta separacion, la segunda corrida reanudaria por error
    desde el estado de la primera, contaminando resultados y tiempos.'''
    sufijo = f"_{tag}" if tag else ""
    return os.path.join(ckpt_dir, f"ckpt_N{N}{sufijo}_seed{seed}.pkl")


def guardar_checkpoint(ruta, estado):
    '''Escritura ATOMICA: se escribe a un archivo temporal y se reemplaza de una sola vez.
    Evita dejar un checkpoint corrupto si el entorno se corta a mitad de la escritura.'''
    tmp = ruta + ".tmp"
    with open(tmp, "wb") as fh:
        pickle.dump(estado, fh)
    os.replace(tmp, ruta)


def cargar_checkpoint(ruta):
    with open(ruta, "rb") as fh:
        return pickle.load(fh)

In [ ]:
def busqueda_local(individuo, max_iter):
    '''Refuerzo memetico (declarado, usado solo para N=50): aplica intercambios dirigidos
    a columnas en conflicto que reducen el fitness. Es una mejora Lamarckiana acotada; el
    operador sigue siendo un swap. Se mantiene throttled para no anular el aporte del AG.'''
    actual = conflictos_diagonales(individuo)
    n = len(individuo)
    for _ in range(max_iter):
        if actual == 0:
            break
        d1 = Counter(); d2 = Counter()
        for c, f in enumerate(individuo):
            d1[f - c] += 1; d2[f + c] += 1
        cols_conf = [c for c, f in enumerate(individuo) if d1[f - c] > 1 or d2[f + c] > 1]
        if not cols_conf:
            break
        i = random.choice(cols_conf)
        mejor, mejor_j = actual, None
        for j in range(n):
            if j == i:
                continue
            individuo[i], individuo[j] = individuo[j], individuo[i]
            f = conflictos_diagonales(individuo)
            individuo[i], individuo[j] = individuo[j], individuo[i]
            if f < mejor:
                mejor, mejor_j = f, j
        if mejor_j is None:
            break
        individuo[i], individuo[mejor_j] = individuo[mejor_j], individuo[i]
        actual = mejor
    return actual

In [ ]:
def run_ga(N, seed, params, ckpt_dir, ckpt_every=25, memetico=False, tag="", verbose=False):
    '''Motor evolutivo manual con elitismo, reinicio por estancamiento, parada temprana por
    optimo conocido y checkpointing atomico reanudable. El `tag` identifica la configuracion
    en el nombre del checkpoint para que corridas distintas con el mismo (N, seed) no
    compartan archivo.'''
    tb = construir_toolbox(N)
    stats = tools.Statistics(lambda ind: ind.fitness.values[0])
    stats.register("min", min); stats.register("avg", lambda x: sum(x) / len(x)); stats.register("max", max)

    ruta = ruta_checkpoint(ckpt_dir, N, seed, tag)
    pop_size = params["pop_size"]; max_gen = params["max_gen"]
    cxpb = params["cxpb"]; mutpb = params["mutpb"]
    n_elite = params["n_elite"]; stagn = params["stagn"]
    ls_pb = params.get("ls_pb", 0.0); ls_iter = params.get("ls_iter", 0)

    if os.path.exists(ruta):
        cp = cargar_checkpoint(ruta)
        random.setstate(cp["rndstate"])            # RESTAURA estado del RNG, no re-siembra
        np.random.set_state(cp["np_rndstate"])
        population = cp["population"]; gen = cp["generation"]
        hof = cp["halloffame"]; logbook = cp["logbook"]
        best = cp["best"]; ultima_mejora = cp["ultima_mejora"]; elapsed = cp["elapsed"]
        if verbose:
            print(f"  [reanudado] N={N} seed={seed} tag={tag} desde gen={gen}")
    else:
        random.seed(seed); np.random.seed(seed)    # se siembra UNA sola vez
        population = tb.population(n=pop_size)
        for ind in population:
            ind.fitness.values = tb.evaluate(ind)
        hof = tools.HallOfFame(1); hof.update(population)
        logbook = tools.Logbook(); logbook.header = ["gen", "min", "avg", "max"]
        rec = stats.compile(population); logbook.record(gen=0, **rec)
        gen = 0; best = hof[0].fitness.values[0]; ultima_mejora = 0; elapsed = 0.0

    t0 = time.time()
    while gen < max_gen and hof[0].fitness.values[0] > 0:
        gen += 1
        elite = [tb.clone(x) for x in tools.selBest(population, n_elite)]
        offspring = [tb.clone(x) for x in tb.select(population, len(population) - n_elite)]
        for c1, c2 in zip(offspring[::2], offspring[1::2]):
            if random.random() < cxpb:
                tb.mate(c1, c2); del c1.fitness.values; del c2.fitness.values
        for m in offspring:
            if random.random() < mutpb:
                tb.mutate(m); del m.fitness.values
        if memetico and ls_pb > 0:
            for m in offspring:
                if random.random() < ls_pb:
                    busqueda_local(m, ls_iter)
                    if m.fitness.valid:
                        del m.fitness.values
        invalidos = [ind for ind in offspring if not ind.fitness.valid]
        for ind in invalidos:
            ind.fitness.values = tb.evaluate(ind)
        population = elite + offspring
        hof.update(population)
        rec = stats.compile(population); logbook.record(gen=gen, **rec)

        actual = hof[0].fitness.values[0]
        if actual < best:
            best = actual; ultima_mejora = gen
        # reinicio por estancamiento: conserva la elite y re-inyecta individuos aleatorios
        if gen - ultima_mejora >= stagn:
            conservados = [tb.clone(x) for x in tools.selBest(population, n_elite)]
            while len(conservados) < pop_size:
                ni = tb.individual(); ni.fitness.values = tb.evaluate(ni)
                conservados.append(ni)
            population = conservados; ultima_mejora = gen

        if gen % ckpt_every == 0 or hof[0].fitness.values[0] == 0:
            estado = {
                "N": N, "seed": seed, "population": population, "generation": gen,
                "halloffame": hof, "logbook": logbook,
                "rndstate": random.getstate(), "np_rndstate": np.random.get_state(),
                "best": best, "ultima_mejora": ultima_mejora,
                "elapsed": elapsed + (time.time() - t0),
            }
            guardar_checkpoint(ruta, estado)

    elapsed += time.time() - t0
    convergio = hof[0].fitness.values[0] == 0
    gen_optimo = gen if convergio else None
    return {
        "N": N, "seed": seed, "convergio": convergio,
        "best_fitness": int(hof[0].fitness.values[0]),
        "generaciones": gen, "gen_optimo": gen_optimo,
        "tiempo_s": round(elapsed, 3),
        "logbook": logbook, "mejor_individuo": list(hof[0]),
    }

print("Motor evolutivo definido.")

Motor evolutivo definido.


## 5. Experimentacion multi-corrida

Una unica corrida no tiene validez analitica. Para cada $N$ se ejecutan varias semillas
independientes y se consolidan los resultados en un DataFrame.

Los hiperparametros escalan con $N$. Para $N=50$ se registran dos configuraciones: el **AG
puro** con los operadores de consigna (que evidencia el comportamiento del metodo a esa
escala) y un **AG con refuerzo memetico** declarado, necesario para obtener efectivamente una
solucion valida de $0$ conflictos en presupuesto razonable.

In [ ]:
# Hiperparametros por configuracion (punto de partida ajustado empiricamente)
CONFIGS = {
    "N8":        dict(N=8,  pop_size=150, max_gen=300, cxpb=0.80, mutpb=0.20, n_elite=2, stagn=60,  memetico=False, R=5),
    "N20":       dict(N=20, pop_size=400, max_gen=1000, cxpb=0.80, mutpb=0.30, n_elite=2, stagn=100, memetico=False, R=5),
    "N50_puro":  dict(N=50, pop_size=600, max_gen=300, cxpb=0.85, mutpb=0.30, n_elite=3, stagn=120, memetico=False, R=3),
    "N50_mem":   dict(N=50, pop_size=400, max_gen=400, cxpb=0.85, mutpb=0.30, n_elite=3, stagn=120, memetico=True,  R=3,
                      ls_pb=0.06, ls_iter=4),
}

tabla_hiper = pd.DataFrame([
    {"Config": k, "N": c["N"], "pop_size": c["pop_size"], "max_gen": c["max_gen"],
     "cxpb": c["cxpb"], "mutpb": c["mutpb"], "tournsize": 3, "memetico": c["memetico"], "R": c["R"]}
    for k, c in CONFIGS.items()
])
print(tabla_hiper.to_string(index=False))

  Config  N  pop_size  max_gen  cxpb  mutpb  tournsize  memetico  R
      N8  8       150      300  0.80    0.2          3     False  5
     N20 20       400     1000  0.80    0.3          3     False  5
N50_puro 50       600      300  0.85    0.3          3     False  3
 N50_mem 50       400      400  0.85    0.3          3      True  3


In [ ]:
def correr_config(nombre, cfg, ckpt_dir):
    params = {k: cfg[k] for k in ["pop_size", "max_gen", "cxpb", "mutpb", "n_elite", "stagn"]}
    if cfg.get("memetico"):
        params["ls_pb"] = cfg["ls_pb"]; params["ls_iter"] = cfg["ls_iter"]
    resultados = []
    for r in range(cfg["R"]):
        seed = SEMILLA_BASE + r
        # tag=nombre separa los checkpoints de cada configuracion (evita que, p. ej.,
        # N50_mem reanude por error desde el checkpoint de N50_puro al compartir (N, seed)).
        res = run_ga(cfg["N"], seed, params, ckpt_dir, ckpt_every=25,
                     memetico=cfg.get("memetico", False), tag=nombre)
        res["config"] = nombre
        resultados.append(res)
        estado = "OK" if res["convergio"] else f"no converge (best={res['best_fitness']})"
        print(f"  {nombre} seed={seed}: gen={res['generaciones']:>4}  t={res['tiempo_s']:>7.2f}s  {estado}")
    return resultados

todos = []
for nombre, cfg in CONFIGS.items():
    print(f"\n=== {nombre} (N={cfg['N']}, R={cfg['R']}) ===")
    todos.extend(correr_config(nombre, cfg, CKPT_DIR))

# DataFrame de resultados (sin el logbook, que se conserva aparte para los graficos)
logbooks = {(r["config"], r["seed"]): r["logbook"] for r in todos}
df = pd.DataFrame([{k: v for k, v in r.items() if k != "logbook"} for r in todos])
df.to_csv(os.path.join(FIGS_DIR, "resultados_corridas.csv"), index=False)
print("\nResultados consolidados:")
print(df[["config", "N", "seed", "convergio", "best_fitness", "generaciones", "tiempo_s"]].to_string(index=False))


=== N8 (N=8, R=5) ===
  N8 seed=42: gen=   2  t=   0.01s  OK
  N8 seed=43: gen=   4  t=   0.01s  OK
  N8 seed=44: gen=   0  t=   0.00s  OK
  N8 seed=45: gen=   0  t=   0.00s  OK
  N8 seed=46: gen=   4  t=   0.01s  OK

=== N20 (N=20, R=5) ===


  N20 seed=42: gen=  50  t=   0.58s  OK


  N20 seed=43: gen=  76  t=   0.94s  OK


  N20 seed=44: gen=  19  t=   0.22s  OK


  N20 seed=45: gen= 129  t=   1.63s  OK


...

  N50_puro seed=43: gen= 300  t=   9.89s  no converge (best=3)


  N50_puro seed=44: gen= 300  t=   9.78s  no converge (best=3)

=== N50_mem (N=50, R=3) ===


  N50_mem seed=42: gen=  15  t=   2.21s  OK


  N50_mem seed=43: gen=  37  t=   4.92s  OK


  N50_mem seed=44: gen=  41  t=   5.64s  OK

Resultados consolidados:
  config  N  seed  convergio  best_fitness  generaciones  tiempo_s
      N8  8    42       True             0             2     0.007
      N8  8    43       True             0             4     0.012
      N8  8    44       True             0             0     0.000
      N8  8    45       True             0             0     0.000
      N8  8    46       True             0             4     0.012
     N20 20    42       True             0            50     0.583
     N20 20    43       True             0            76     0.936
     N20 20    44       True             0            19     0.224
     N20 20    45       True             0           129     1.627
     N20 20    46       True             0           385     4.785
N50_puro 50    42      False             6           300     9.966
N50_puro 50    43      False             3           300     9.890
N50_puro 50    44      False             3           300   

## 6. Analisis

### 6.1 Algoritmo genetico frente a la busqueda exhaustiva

Se compara el tiempo real del AG contra el de una busqueda por fuerza bruta que evalua cada
permutacion a razon de $10^{-6}$ s. Para fijar un umbral de viabilidad "en tiempo humano" se
adopta un limite tolerable de **1 hora** ($3600$ s), equivalente a $3.6\times10^{9}$
permutaciones; el menor $N$ cuyo $N!$ supera ese limite marca el punto en que la busqueda
exhaustiva deja de ser practicable.

In [ ]:
SEG_POR_PERM = 1e-6
LIMITE_HUMANO_S = 3600  # 1 hora

def humano(seg):
    if seg < 60: return f"{seg:.2f} s"
    if seg < 3600: return f"{seg/60:.1f} min"
    if seg < 86400: return f"{seg/3600:.1f} h"
    if seg < 3.15e7: return f"{seg/86400:.1f} dias"
    return f"{seg/3.15e7:.3e} anios"

# tiempo real promedio del AG por N (para N=50 se toma la config que efectivamente converge)
def tiempo_ag(config):
    sub = df[df["config"] == config]
    conv = sub[sub["convergio"]]
    base = conv if len(conv) else sub
    return base["tiempo_s"].mean()

filas = []
for N, cfg_name in [(8, "N8"), (20, "N20"), (50, "N50_mem")]:
    permut = math.factorial(N)
    t_fb = permut * SEG_POR_PERM
    filas.append({
        "N": N,
        "N!": f"{permut:.3e}",
        "Fuerza bruta (10^-6 s/perm)": humano(t_fb),
        "AG - tiempo real (prom)": f"{tiempo_ag(cfg_name):.3f} s",
    })
tabla_fb = pd.DataFrame(filas)
print(tabla_fb.to_string(index=False))

# Umbral de inviabilidad
N_umbral = next(n for n in range(4, 40) if math.factorial(n) * SEG_POR_PERM > LIMITE_HUMANO_S)
print(f"\nUmbral (limite {LIMITE_HUMANO_S} s): la busqueda exhaustiva deja de ser viable a partir de N = {N_umbral}")
print(f"  {N_umbral-1}! = {math.factorial(N_umbral-1):.3e} -> {humano(math.factorial(N_umbral-1)*SEG_POR_PERM)}")
print(f"  {N_umbral}! = {math.factorial(N_umbral):.3e} -> {humano(math.factorial(N_umbral)*SEG_POR_PERM)}")
tabla_fb.to_csv(os.path.join(FIGS_DIR, "tabla_fuerzabruta_vs_ag.csv"), index=False)

 N        N! Fuerza bruta (10^-6 s/perm) AG - tiempo real (prom)
 8 4.032e+04                      0.04 s                 0.006 s
20 2.433e+18             7.723e+04 anios                 1.631 s
50 3.041e+64             9.655e+50 anios                 4.256 s

Umbral (limite 3600 s): la busqueda exhaustiva deja de ser viable a partir de N = 13
  12! = 4.790e+08 -> 8.0 min
  13! = 6.227e+09 -> 1.7 h


### 6.2 Soluciones multiples equivalentes

El problema admite muchas soluciones optimas con identico fitness ($N=8$ tiene $92$;
$N=20$ supera los dos millones). Por tratarse de un problema de *factibilidad*, alcanza con
hallar **una** solucion valida: buscar todas multiplicaria el costo sin aportar valor.

Esto tiene dos consecuencias practicas. Primero, **habilita la parada temprana**: en cuanto
una corrida alcanza fitness $0$ se detiene. Segundo, **define como comparar corridas**:
distintas semillas encuentran tableros distintos pero igualmente validos, de modo que la
comparacion debe hacerse sobre *tiempo* y *generaciones hasta el optimo*, nunca sobre cual
configuracion especifica de tablero se obtuvo.

In [ ]:
# Evidencia: las semillas que convergen para un mismo N producen tableros distintos
conv8 = df[(df["config"] == "N8") & (df["convergio"])]
print("Soluciones validas distintas halladas para N=8 (una por semilla):")
vistas = set()
for _, row in conv8.iterrows():
    ind = next(r["mejor_individuo"] for r in todos if r["config"] == "N8" and r["seed"] == row["seed"])
    vistas.add(tuple(ind))
    print(f"  seed={row['seed']}: {ind}")
print(f"\nTableros distintos: {len(vistas)} de {len(conv8)} corridas convergentes (mismo fitness 0).")

Soluciones validas distintas halladas para N=8 (una por semilla):
  seed=42: [2, 6, 1, 7, 4, 0, 3, 5]
  seed=43: [5, 2, 6, 1, 7, 4, 0, 3]
  seed=44: [1, 4, 6, 3, 0, 7, 5, 2]
  seed=45: [4, 0, 3, 5, 7, 1, 6, 2]
  seed=46: [6, 2, 0, 5, 7, 4, 1, 3]

Tableros distintos: 5 de 5 corridas convergentes (mismo fitness 0).


### 6.3 Escalabilidad

Se caracteriza como crecen poblacion y generaciones requeridas con $N$, usando los datos
reales de las corridas. La columna de convergencia distingue el comportamiento del AG puro
a cada escala.

In [ ]:
filas = []
for nombre, cfg in CONFIGS.items():
    sub = df[df["config"] == nombre]
    conv = sub[sub["convergio"]]
    filas.append({
        "Config": nombre,
        "N": cfg["N"],
        "pop_size": cfg["pop_size"],
        "Tasa exito": f"{len(conv)}/{len(sub)}",
        "Gen. media (converg.)": f"{conv['generaciones'].mean():.0f}" if len(conv) else "-",
        "Tiempo medio (s)": f"{sub['tiempo_s'].mean():.2f}",
    })
tabla_escal = pd.DataFrame(filas)
print(tabla_escal.to_string(index=False))
tabla_escal.to_csv(os.path.join(FIGS_DIR, "tabla_escalabilidad.csv"), index=False)

print("\nLectura del escalado (AG puro): de N=8 a N=20 las generaciones crecen de forma marcada;")
print("a N=50 el AG puro deja de converger en el presupuesto, lo que evidencia un crecimiento")
print("super-lineal del esfuerzo y el limite practico del metodo a esa escala.")

  Config  N  pop_size Tasa exito Gen. media (converg.) Tiempo medio (s)
      N8  8       150        5/5                     2             0.01
     N20 20       400        5/5                   132             1.63
N50_puro 50       600        0/3                     -             9.88
 N50_mem 50       400        3/3                    31             4.26

Lectura del escalado (AG puro): de N=8 a N=20 las generaciones crecen de forma marcada;
a N=50 el AG puro deja de converger en el presupuesto, lo que evidencia un crecimiento
super-lineal del esfuerzo y el limite practico del metodo a esa escala.


### 6.4 Curvas de convergencia

Se grafica el fitness minimo por generacion (del logbook) mostrando el descenso hacia $0$:
una caida rapida inicial y una cola mas lenta cerca del optimo.

In [ ]:
def curva_min(logbook):
    gens = logbook.select("gen")
    mins = logbook.select("min")
    return gens, mins

# Una figura por configuracion representativa (semilla base)
def plot_convergencia(config, titulo, archivo):
    lb = logbooks[(config, SEMILLA_BASE)]
    gens, mins = curva_min(lb)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(gens, mins, lw=1.6)
    ax.set_xlabel("Generacion"); ax.set_ylabel("Conflictos diagonales (min)")
    ax.set_title(titulo); ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGS_DIR, archivo), dpi=150)
    plt.close(fig)
    print("Guardado:", archivo)

plot_convergencia("N8",  "Convergencia AG - N=8",  "fig_convergencia_N8.png")
plot_convergencia("N20", "Convergencia AG - N=20", "fig_convergencia_N20.png")
plot_convergencia("N50_puro", "AG puro - N=50 (no converge en presupuesto)", "fig_convergencia_N50_puro.png")
plot_convergencia("N50_mem",  "AG con refuerzo memetico - N=50", "fig_convergencia_N50_memetico.png")

# Figura comparada (log) de las cuatro curvas. N=50 puro y memetico se estancan en el mismo
# nivel (~6 conflictos) antes de separarse, por lo que se distinguen con estilo de linea: el
# AG puro se traza punteado y mas grueso, dibujado en ultimo lugar para que quede visible
# por encima de la curva memetica en el tramo donde ambas se superponen.
series = [
    ("N8",       "N=8",                      dict(lw=1.6, ls="-",  color="C0")),
    ("N20",      "N=20",                     dict(lw=1.6, ls="-",  color="C1")),
    ("N50_mem",  "N=50 (memetico)",          dict(lw=1.8, ls="-",  color="C3")),
    ("N50_puro", "N=50 (puro, no converge)", dict(lw=2.4, ls="--", color="C2", alpha=0.95)),
]
fig, ax = plt.subplots(figsize=(8, 4.5))
for config, etiqueta, estilo in series:
    gens, mins = curva_min(logbooks[(config, SEMILLA_BASE)])
    ax.plot(gens, mins, label=etiqueta, **estilo)
ax.set_yscale("symlog")
ax.set_xlabel("Generacion"); ax.set_ylabel("Conflictos diagonales (min, symlog)")
ax.set_title("Curvas de convergencia comparadas"); ax.grid(alpha=0.3); ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(FIGS_DIR, "fig_convergencia_comparada.png"), dpi=150)
plt.close(fig)
print("Guardado: fig_convergencia_comparada.png")

Guardado: fig_convergencia_N8.png


Guardado: fig_convergencia_N20.png


Guardado: fig_convergencia_N50_puro.png


Guardado: fig_convergencia_N50_memetico.png


Guardado: fig_convergencia_comparada.png


### 6.5 Visualizacion de tablero

Se representa una solucion valida sobre el tablero de ajedrez. Cada reina ocupa una fila y una
columna distintas; al tener fitness $0$, tampoco comparten diagonal.

In [ ]:
def dibujar_tablero(perm, titulo, archivo):
    N = len(perm)
    lado = min(9.0, max(5.0, 0.6 + N * 0.16))
    fig, ax = plt.subplots(figsize=(lado, lado))
    tablero = np.add.outer(np.arange(N), np.arange(N)) % 2
    ax.imshow(tablero, cmap="binary", alpha=0.25, origin="lower",
              extent=(-0.5, N - 0.5, -0.5, N - 0.5))
    fontsize = max(7, min(22, int(260 / N)))
    for col, fila in enumerate(perm):
        ax.text(col, fila, "Q", ha="center", va="center",
                fontsize=fontsize, fontweight="bold", color="C3")
    if N <= 20:
        ax.set_xticks(range(N)); ax.set_yticks(range(N))
    else:
        ax.set_xticks(range(0, N, 5)); ax.set_yticks(range(0, N, 5))
    ax.set_xlabel("Columna"); ax.set_ylabel("Fila")
    ax.set_title(titulo)
    ax.set_xlim(-0.5, N - 0.5); ax.set_ylim(-0.5, N - 0.5)
    fig.tight_layout()
    fig.savefig(os.path.join(FIGS_DIR, archivo), dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Guardado: {archivo} (fitness = {conflictos_diagonales(perm)})")

# Mejor solucion de N=8 (semilla base)
sol8 = next(r["mejor_individuo"] for r in todos if r["config"] == "N8" and r["seed"] == SEMILLA_BASE)
dibujar_tablero(sol8, "Solucion valida N=8", "fig_tablero_N8.png")

# Solucion de N=50 (config memetica, primera semilla que converge)
sol50_row = df[(df["config"] == "N50_mem") & (df["convergio"])].head(1)
if len(sol50_row):
    s50 = int(sol50_row.iloc[0]["seed"])
    sol50 = next(r["mejor_individuo"] for r in todos if r["config"] == "N50_mem" and r["seed"] == s50)
    dibujar_tablero(sol50, "Solucion valida N=50 (refuerzo memetico)", "fig_tablero_N50.png")

Guardado: fig_tablero_N8.png (fitness = 0)


Guardado: fig_tablero_N50.png (fitness = 0)


## 7. Validacion de reproducibilidad y reanudacion

Se demuestra que el sistema de checkpoints reproduce exactamente una corrida interrumpida.
La prueba: ejecutar una corrida completa sin cortes y, por separado, ejecutar la misma
semilla en dos tramos (cortando a mitad y reanudando desde el checkpoint). Ambos finales
deben coincidir en generacion de convergencia y mejor fitness, porque se restaura el estado
del RNG en lugar de re-sembrar.

In [ ]:
import shutil

def corrida_completa(N, seed, params, base_dir):
    d = os.path.join(base_dir, "prueba_completa")
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
    return run_ga(N, seed, params, d, ckpt_every=10)

def corrida_en_dos_tramos(N, seed, params, base_dir, corte):
    d = os.path.join(base_dir, "prueba_reanudada")
    if os.path.exists(d): shutil.rmtree(d)
    os.makedirs(d, exist_ok=True)
    # tramo 1: presupuesto recortado -> deja checkpoint
    p1 = dict(params); p1["max_gen"] = corte
    run_ga(N, seed, p1, d, ckpt_every=10)
    # tramo 2: reanuda desde checkpoint con presupuesto completo
    return run_ga(N, seed, params, d, ckpt_every=10)

# Se usa N=20 (converge y es suficientemente largo para cortar a mitad)
params_test = dict(pop_size=400, max_gen=1000, cxpb=0.80, mutpb=0.30, n_elite=2, stagn=100)
res_full = corrida_completa(20, SEMILLA_BASE, params_test, CKPT_DIR)
res_resume = corrida_en_dos_tramos(20, SEMILLA_BASE, params_test, CKPT_DIR, corte=20)

print("Corrida completa  : gen_optimo =", res_full["gen_optimo"], "| best =", res_full["best_fitness"],
      "| individuo =", res_full["mejor_individuo"][:8], "...")
print("Corrida reanudada : gen_optimo =", res_resume["gen_optimo"], "| best =", res_resume["best_fitness"],
      "| individuo =", res_resume["mejor_individuo"][:8], "...")

iguales = (res_full["gen_optimo"] == res_resume["gen_optimo"]
           and res_full["mejor_individuo"] == res_resume["mejor_individuo"])
print("\nReanudacion reproduce exactamente la corrida sin interrupcion:", iguales)
assert iguales, "La reanudacion no coincide con la corrida completa"


Corrida completa  : gen_optimo = 50 | best = 0 | individuo = [2, 9, 1, 19, 8, 15, 11, 14] ...
Corrida reanudada : gen_optimo = 50 | best = 0 | individuo = [2, 9, 1, 19, 8, 15, 11, 14] ...

Reanudacion reproduce exactamente la corrida sin interrupcion: True


In [ ]:
# Prueba multi-semilla: trayectorias distintas, todas validas cuando convergen
print("Prueba multi-semilla (N=20):")
sub = df[df["config"] == "N20"]
for _, row in sub.iterrows():
    print(f"  seed={int(row['seed'])}: gen={int(row['generaciones'])}, "
          f"convergio={bool(row['convergio'])}, best={int(row['best_fitness'])}")
print(f"\nConvergen {int(sub['convergio'].sum())}/{len(sub)} semillas, cada una a un tablero valido distinto.")

Prueba multi-semilla (N=20):
  seed=42: gen=50, convergio=True, best=0
  seed=43: gen=76, convergio=True, best=0
  seed=44: gen=19, convergio=True, best=0
  seed=45: gen=129, convergio=True, best=0
  seed=46: gen=385, convergio=True, best=0

Convergen 5/5 semillas, cada una a un tablero valido distinto.


## 8. Exportacion de figuras para el informe

Se empaquetan todas las figuras y tablas generadas en un unico archivo ZIP y se ofrece su
descarga. En Colab la descarga es directa; fuera de Colab queda el ZIP en el directorio de
trabajo.

In [ ]:
zip_path = "figuras_informe_NReinas.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for nombre in sorted(os.listdir(FIGS_DIR)):
        zf.write(os.path.join(FIGS_DIR, nombre), arcname=nombre)

print("Contenido del ZIP:")
with zipfile.ZipFile(zip_path) as zf:
    for n in zf.namelist():
        print("  -", n)
print("\nZIP generado:", os.path.abspath(zip_path))

Contenido del ZIP:
  - fig_convergencia_N20.png
  - fig_convergencia_N50_memetico.png
  - fig_convergencia_N50_puro.png
  - fig_convergencia_N8.png
  - fig_convergencia_comparada.png
  - fig_tablero_N50.png
  - fig_tablero_N8.png
  - resultados_corridas.csv
  - tabla_escalabilidad.csv
  - tabla_fuerzabruta_vs_ag.csv

ZIP generado: /tmp/figuras_informe_NReinas.zip


In [ ]:
# Descarga del ZIP (directa en Colab; fuera de Colab informa la ruta local)
try:
    from google.colab import files  # type: ignore
    files.download(zip_path)
except Exception:
    print("Fuera de Colab: el archivo esta disponible en", os.path.abspath(zip_path))

Fuera de Colab: el archivo esta disponible en /tmp/figuras_informe_NReinas.zip
